# ZooPredict: Zoonotic Disease Outbreak Risk Prediction

**Author:** Karimat Opeyemi Abolarinwa  
**Programme:** DSHub AI/ML Engineering — Group AI4, Cohort A 2026  
**Background:** BSc Zoology (OAU) + ML Engineering  

---

### Overview

Zoonotic diseases — infections that jump from animals to humans — account for the majority of emerging infectious disease events globally. Sub-Saharan Africa, with its wildlife biodiversity and variable healthcare infrastructure, bears a disproportionate burden.

This notebook builds a multi-class ML classifier that predicts outbreak risk levels (Low / Medium / High) at the country-month level, using wildlife surveillance data, climate variables, and socioeconomic indicators. The goal is a decision-support tool that helps field surveillance teams prioritise high-risk locations.

**Why this matters to me:** My BSc in Zoology at OAU covered animal ecology and host-pathogen relationships. The 2014 Ebola outbreak showed me that understanding animal reservoirs has direct public health implications. This project connects that background with ML.

---

### Objectives

1. Build a multi-class outbreak risk classifier using ensemble methods
2. Identify the most significant ecological, climatic, and socioeconomic predictors
3. Compare Logistic Regression, Random Forest, Gradient Boosting, and XGBoost
4. Derive interpretable policy insights for surveillance prioritisation

---

### Dataset

Since WHO/CDC case-level surveillance microdata isn't publicly available, this project uses a synthetic dataset that mirrors the structure and distributions of real WHO/PREDICT surveillance records. All features are grounded in published epidemiological literature (Olival et al. 2017, Jones et al. 2008).

**Target variable:** `outbreak_risk_level` — Low / Medium / High

## Section 1: Setup

In [1]:
# Install required libraries (run this cell first in Google Colab)
# If running locally and packages are already installed, this cell can be skipped

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

packages = ['xgboost', 'seaborn', 'plotly', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'scipy']
for p in packages:
    try:
        __import__(p.replace('-','_'))
    except ImportError:
        print(f'Installing {p}...')
        install(p)

print('All packages ready.')


Installing xgboost...


CalledProcessError: Command '['c:\\Users\\Karimat Odewale\\AppData\\Local\\Programs\\Python\\Python311\\python.exe', '-m', 'pip', 'install', 'xgboost', '-q']' returned non-zero exit status 1.

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              ConfusionMatrixDisplay, accuracy_score)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
import xgboost as xgb
from scipy import stats

# Plotting config
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold'
})
COLORS = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
PALETTE = ['#2ecc71', '#f39c12', '#e74c3c']

# Reproducibility
np.random.seed(42)

print("Libraries loaded successfully.")
print(f"   Pandas: {pd.__version__} | NumPy: {np.__version__} | XGBoost: {xgb.__version__}")


## Section 2: Case Study

**Scenario:** A fictional public health agency (AOHIN) monitors zoonotic disease risk across 16 Sub-Saharan African countries. They collect monthly environmental, wildlife, and healthcare data but can't inspect every location every month. They need a model to rank locations by risk level so they can allocate limited field teams.

**Diseases:** Ebola, Lassa Fever, Monkeypox, Avian Influenza, Rift Valley Fever, Nipah, Marburg, Rabies, Plague, West Nile Fever

Risk labels are derived from a weighted composite model based on Olival et al. (2017) and Jones et al. (2008).

## Section 3: Dataset Construction

In [ ]:
# I build a richly structured synthetic surveillance dataset mirroring
# real WHO/PREDICT-style zoonotic surveillance records.
# Each row = one country-month observation.

np.random.seed(42)
N = 2000  # 2,000 surveillance records

# Geographic and biological categorical variables
COUNTRIES = [
    'Nigeria', 'Ghana', 'Democratic Republic of Congo', 'Kenya', 'Ethiopia',
    'Cameroon', 'Uganda', 'Tanzania', 'Senegal', 'Mali',
    'Guinea', 'Sierra Leone', 'Liberia', "Côte d'Ivoire", 'Mozambique', 'Zimbabwe'
]
WILDLIFE_HOSTS = ['Bat', 'Rodent', 'Primate', 'Bird', 'Swine', 'Camel', 'Unknown']
DISEASES = [
    'Ebola', 'Lassa Fever', 'Monkeypox', 'Avian Influenza',
    'Rift Valley Fever', 'Nipah', 'Marburg', 'Rabies', 'Plague', 'West Nile Fever'
]

# Continuous environmental & socioeconomic features
country           = np.random.choice(COUNTRIES, N)
host_species      = np.random.choice(WILDLIFE_HOSTS, N)
disease_reported  = np.random.choice(DISEASES, N)
temperature       = np.random.normal(27, 5, N)                   # °C
rainfall_mm       = np.random.exponential(120, N)                # mm/month
humidity_pct      = np.random.uniform(40, 95, N)                 # %
pop_density       = np.random.exponential(150, N)                # persons/km²
deforestation     = np.random.uniform(0, 12, N)                  # % annual loss
livestock_density = np.random.exponential(50, N)                 # livestock units/km²
wildlife_trade    = np.random.uniform(0, 10, N)                  # index 0–10
healthcare_idx    = np.random.uniform(10, 80, N)                 # score 0–100
surveillance_cov  = np.random.uniform(0.1, 1.0, N)              # fraction 0–1
forest_proximity  = np.random.uniform(0, 500, N)                 # km
prev_outbreaks    = np.random.poisson(1.5, N)                    # count
travel_connect    = np.random.uniform(0, 100, N)                 # index
wet_season        = np.random.choice([0, 1], N, p=[0.45, 0.55])
sanitation_idx    = np.random.uniform(10, 90, N)                 # score 0–100
month             = np.random.randint(1, 13, N)

# Each weight is grounded in epidemiological evidence:

risk_score = (
    0.15 * (temperature - 20) / 20 +
    0.12 * (rainfall_mm / 200) +
    0.10 * (deforestation / 12) +
    0.12 * (pop_density / 300) +
    0.08 * (wildlife_trade / 10) +
    0.10 * (1 - healthcare_idx / 80) +
    0.08 * (1 - surveillance_cov) +
    0.07 * (prev_outbreaks / 5) +
    0.06 * (livestock_density / 100) +
    0.05 * wet_season +
    0.04 * (1 - sanitation_idx / 90) +
    0.03 * (forest_proximity < 50).astype(int)
) + np.random.normal(0, 0.08, N)

def assign_risk(score):
    if score < 0.28:   return 'Low'
    elif score < 0.52: return 'Medium'
    else:              return 'High'

outbreak_risk = [assign_risk(s) for s in risk_score]

df = pd.DataFrame({
    'country':                    country,
    'host_species':               host_species,
    'disease_reported':           disease_reported,
    'mean_temperature_c':         temperature.round(2),
    'rainfall_mm':                rainfall_mm.round(2),
    'humidity_pct':               humidity_pct.round(2),
    'human_pop_density_km2':      pop_density.round(2),
    'deforestation_rate_pct':     deforestation.round(3),
    'livestock_density':          livestock_density.round(2),
    'wildlife_trade_index':       wildlife_trade.round(2),
    'healthcare_index':           healthcare_idx.round(2),
    'surveillance_coverage':      surveillance_cov.round(3),
    'proximity_to_forest_km':     forest_proximity.round(2),
    'prev_outbreaks_5yr':         prev_outbreaks,
    'travel_connectivity':        travel_connect.round(2),
    'wet_season':                 wet_season,
    'sanitation_index':           sanitation_idx.round(2),
    'month':                      month,
    'outbreak_risk_level':        outbreak_risk
})

print(f"Dataset created: {df.shape[0]:,} records × {df.shape[1]} features")
print(f"   Missing values: {df.isnull().sum().sum()}")
print()
print("Class distribution:")
print(df['outbreak_risk_level'].value_counts().to_string())


In [ ]:
# Preview the dataset
df.head(8)


In [ ]:
# Statistical summary of continuous features
df.describe().round(2)


## Section 4: Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['outbreak_risk_level'].value_counts().reindex(['Low','Medium','High'])
bars = axes[0].bar(counts.index, counts.values, color=PALETTE, edgecolor='white', linewidth=1.5, width=0.6)
axes[0].set_title('Outbreak Risk Level Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Risk Level')
axes[0].set_ylabel('Number of Observations')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
                 f'{val}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=10, fontweight='bold')

# Pie
axes[1].pie(counts.values, labels=counts.index, colors=PALETTE,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11},
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Risk Level Proportions', fontsize=14, fontweight='bold')

plt.suptitle('Figure 1: Target Variable Overview', fontsize=15, y=1.02, fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Class imbalance note: Medium risk dominates (59%). We handle this via stratified splits and class_weight in models.")


In [ ]:
climate_features = ['mean_temperature_c', 'rainfall_mm', 'humidity_pct']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, feat in zip(axes, climate_features):
    risk_order = ['Low', 'Medium', 'High']
    data_by_risk = [df[df['outbreak_risk_level']==r][feat].values for r in risk_order]
    bp = ax.boxplot(data_by_risk, patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticklabels(risk_order, fontsize=11)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.set_ylabel(feat.split('_')[-1].upper())

    # ANOVA test
    f_stat, p_val = stats.f_oneway(*data_by_risk)
    significance = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))
    ax.text(0.97, 0.96, f'F={f_stat:.1f} {significance}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, color='#333')

plt.suptitle('Figure 2: Climate Feature Distributions by Outbreak Risk Level\n(ANOVA significance: ***p<0.001)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_climate_features.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
socio_features = ['healthcare_index', 'surveillance_coverage',
                   'deforestation_rate_pct', 'sanitation_index',
                   'wildlife_trade_index', 'prev_outbreaks_5yr']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for ax, feat in zip(axes, socio_features):
    risk_order = ['Low', 'Medium', 'High']
    data_by_risk = [df[df['outbreak_risk_level']==r][feat].values for r in risk_order]
    bp = ax.boxplot(data_by_risk, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticklabels(risk_order, fontsize=10)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    f_stat, p_val = stats.f_oneway(*data_by_risk)
    sig = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))
    ax.text(0.97, 0.96, f'p {sig}', transform=ax.transAxes, ha='right', va='top',
            fontsize=9, style='italic', color='#555')

plt.suptitle('Figure 3: Socioeconomic & Ecological Features by Risk Level', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_socio_features.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

cat_features = ['host_species', 'country', 'wet_season']
cat_labels   = ['Wildlife Host Species', 'Country', 'Wet Season Active']

for ax, feat, label in zip(axes, cat_features, cat_labels):
    ct = pd.crosstab(df[feat], df['outbreak_risk_level'],
                     normalize='index')[['Low','Medium','High']] * 100
    ct.plot(kind='bar', ax=ax, color=PALETTE, edgecolor='white', linewidth=0.8, width=0.75)
    ax.set_title(f'Risk Distribution by {label}', fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('% of observations')
    ax.legend(title='Risk', fontsize=9, loc='upper right')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Figure 4: Outbreak Risk by Categorical Features', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig4_categorical.png', dpi=150, bbox_inches='tight')
plt.show()
print("Key observation: Bat and Primate hosts, DRC and Guinea, and wet season periods show elevated High-risk proportions.")


In [ ]:
numeric_cols = ['mean_temperature_c','rainfall_mm','humidity_pct','human_pop_density_km2',
                'deforestation_rate_pct','livestock_density','wildlife_trade_index',
                'healthcare_index','surveillance_coverage','proximity_to_forest_km',
                'prev_outbreaks_5yr','travel_connectivity','wet_season','sanitation_index','month']

corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.3, annot_kws={'size': 8},
            vmin=-1, vmax=1)
ax.set_title('Figure 5: Feature Correlation Matrix\n(Lower triangle — Pearson r)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_correlation.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
seasonal = df.groupby('month')['outbreak_risk_level'].apply(
    lambda x: (x=='High').mean() * 100).reset_index()
seasonal.columns = ['month', 'pct_high_risk']

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(1,13), seasonal['pct_high_risk'],
              color=['#e74c3c' if v > seasonal['pct_high_risk'].mean() else '#3498db'
                     for v in seasonal['pct_high_risk']],
              edgecolor='white', linewidth=1)
ax.axhline(seasonal['pct_high_risk'].mean(), color='black', linestyle='--',
           linewidth=1.5, label=f"Mean: {seasonal['pct_high_risk'].mean():.1f}%")
ax.set_xticks(range(1,13))
ax.set_xticklabels(month_names)
ax.set_title('Figure 6: Seasonal Distribution of High-Risk Outbreak Months', fontsize=13, fontweight='bold')
ax.set_ylabel('% of Observations Classified as High Risk')
ax.set_xlabel('Month')
ax.legend()
plt.tight_layout()
plt.savefig('fig6_seasonal.png', dpi=150, bbox_inches='tight')
plt.show()
print("Observation: Higher risk concentrations align with wet season months (April–October in West Africa).")


## Section 5: Feature Engineering & Preprocessing

In [ ]:
le_country = LabelEncoder()
le_host    = LabelEncoder()
le_disease = LabelEncoder()
le_target  = LabelEncoder()

df['country_enc']  = le_country.fit_transform(df['country'])
df['host_enc']     = le_host.fit_transform(df['host_species'])
df['disease_enc']  = le_disease.fit_transform(df['disease_reported'])
df['risk_encoded'] = le_target.fit_transform(df['outbreak_risk_level'])
# Mapping: 0=High, 1=Low, 2=Medium — check:
print("Label encoding:", {k:v for k,v in zip(le_target.classes_, le_target.transform(le_target.classes_))})

df['temp_x_rain']          = df['mean_temperature_c'] * df['rainfall_mm'] / 1000
df['deforest_x_pop']       = df['deforestation_rate_pct'] * df['human_pop_density_km2'] / 100
df['surveillance_deficit']  = (1 - df['surveillance_coverage']) * (1 - df['healthcare_index']/100)
df['wildlife_access_score'] = df['wildlife_trade_index'] * (500 - df['proximity_to_forest_km']) / 500

FEATURE_COLS = [
    'mean_temperature_c', 'rainfall_mm', 'humidity_pct',
    'human_pop_density_km2', 'deforestation_rate_pct', 'livestock_density',
    'wildlife_trade_index', 'healthcare_index', 'surveillance_coverage',
    'proximity_to_forest_km', 'prev_outbreaks_5yr', 'travel_connectivity',
    'wet_season', 'sanitation_index', 'month',
    'country_enc', 'host_enc', 'disease_enc',
    'temp_x_rain', 'deforest_x_pop', 'surveillance_deficit', 'wildlife_access_score'
]

TARGET_COL = 'risk_encoded'

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

print(f"\nFeature matrix: {X.shape}")
print(f"   Target: {y.shape} | Classes: {le_target.classes_}")
print(f"   No. of features (incl. engineered): {X.shape[1]}")


In [ ]:
# Stratified split ensures proportional class representation in each set
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print(f"Training set  : {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Validation set: {X_val.shape[0]:,} samples ({X_val.shape[0]/len(X)*100:.0f}%)")
print(f"Test set      : {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.0f}%)")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)
print("\nFeatures scaled (StandardScaler fit on train, applied to val/test).")


## Section 6: Model Training & Evaluation

In [ ]:
print("Training Model 1: Logistic Regression (Baseline)...")
lr_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42, class_weight='balanced')
lr_model.fit(X_train_sc, y_train)
lr_pred  = lr_model.predict(X_test_sc)
lr_acc   = accuracy_score(y_test, lr_pred)
print(f"  Test Accuracy: {lr_acc:.4f}")
print()
print(classification_report(y_test, lr_pred,
      target_names=le_target.classes_, digits=4))


In [ ]:
print("Training Model 2: Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_split=5,
    class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred  = rf_model.predict(X_test)
rf_acc   = accuracy_score(y_test, rf_pred)
print(f"  Test Accuracy: {rf_acc:.4f}")
print()
print(classification_report(y_test, rf_pred,
      target_names=le_target.classes_, digits=4))


In [ ]:
print("Training Model 3: Gradient Boosting Classifier...")
gb_model = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=5,
    subsample=0.8, random_state=42)
gb_model.fit(X_train, y_train)
gb_pred  = gb_model.predict(X_test)
gb_acc   = accuracy_score(y_test, gb_pred)
print(f"  Test Accuracy: {gb_acc:.4f}")
print()
print(classification_report(y_test, gb_pred,
      target_names=le_target.classes_, digits=4))


In [ ]:
print("Training Model 4: XGBoost Classifier (Primary Model)...")

# Compute class weights for imbalanced dataset
class_counts = np.bincount(y_train)
total = len(y_train)
sample_weights = np.array([total / (len(class_counts) * class_counts[yi]) for yi in y_train])

xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.08,
    subsample=0.8, colsample_bytree=0.8, gamma=1,
    reg_alpha=0.1, reg_lambda=1.0,
    eval_metric='mlogloss', random_state=42, verbosity=0
)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights,
              eval_set=[(X_val, y_val)], verbose=False)

xgb_pred     = xgb_model.predict(X_test)
xgb_pred_proba = xgb_model.predict_proba(X_test)
xgb_acc      = accuracy_score(y_test, xgb_pred)
xgb_auc      = roc_auc_score(y_test, xgb_pred_proba, multi_class='ovr', average='weighted')

print(f"  Test Accuracy: {xgb_acc:.4f}")
print(f"  ROC-AUC (weighted OvR): {xgb_auc:.4f}")
print()
print(classification_report(y_test, xgb_pred,
      target_names=le_target.classes_, digits=4))


In [ ]:
print("Running 5-fold Stratified Cross-Validation on all models...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
models_cv = {
    'Logistic Regression': (lr_model, X_train_sc, True),
    'Random Forest':       (rf_model, X_train, False),
    'Gradient Boosting':   (gb_model, X_train, False),
    'XGBoost':             (xgb_model, X_train, False),
}

for name, (model, Xm, scaled) in models_cv.items():
    X_cv = Xm if not scaled else X_train_sc
    scores = cross_val_score(model, X_cv, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    results[name] = scores
    print(f"  {name:<22}: {scores.mean():.4f} ± {scores.std():.4f}  (all folds: {scores.round(3)})")

print("\nCross-validation complete.")


In [ ]:
test_accs = {
    'Logistic\nRegression': lr_acc,
    'Random\nForest':       rf_acc,
    'Gradient\nBoosting':   gb_acc,
    'XGBoost':               xgb_acc
}

cv_means = {k: v.mean() for k, v in results.items()}
cv_stds  = {k: v.std()  for k, v in results.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test accuracy bar chart
model_names = list(test_accs.keys())
acc_vals = list(test_accs.values())
colors_bar = ['#bdc3c7', '#3498db', '#9b59b6', '#e74c3c']
bars = axes[0].bar(model_names, acc_vals, color=colors_bar, edgecolor='white', linewidth=1.5, width=0.6)
for bar, val in zip(bars, acc_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
                 f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylim(0.5, 0.85)
axes[0].set_title('Test Set Accuracy by Model', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Accuracy')
axes[0].axhline(0.7, color='gray', linestyle='--', alpha=0.5, linewidth=1, label='Baseline target 0.70')
axes[0].legend(fontsize=9)

# CV boxplot
cv_data = [results[k] for k in results]
bp = axes[1].boxplot(cv_data, patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], colors_bar):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[1].set_xticklabels(['LR', 'RF', 'GB', 'XGB'], fontsize=11)
axes[1].set_title('5-Fold CV Accuracy Distribution', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Cross-Validation Accuracy')

plt.suptitle('Figure 7: Model Performance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig7_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
class_names = le_target.classes_
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (name, pred) in zip(axes, [
    ('Logistic Regression', lr_pred),
    ('Random Forest',       rf_pred),
    ('Gradient Boosting',   gb_pred),
    ('XGBoost',             xgb_pred)
]):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAcc: {accuracy_score(y_test, pred):.3f}',
                 fontsize=10, fontweight='bold')

plt.suptitle('Figure 8: Confusion Matrices — All Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig8_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 7: Feature Importance & Interpretability

In [ ]:
importances = xgb_model.feature_importances_
feat_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=True)

# Colour by feature category
def categorise(feat):
    if feat in ['mean_temperature_c','rainfall_mm','humidity_pct','wet_season','month']:
        return 'Climate'
    elif feat in ['deforestation_rate_pct','proximity_to_forest_km','wildlife_trade_index',
                  'host_enc','livestock_density']:
        return 'Ecological'
    elif feat in ['healthcare_index','surveillance_coverage','sanitation_index','prev_outbreaks_5yr']:
        return 'Healthcare/Surveillance'
    elif feat in ['human_pop_density_km2','travel_connectivity','country_enc','disease_enc']:
        return 'Socioeconomic'
    else:
        return 'Engineered'

cat_colors = {
    'Climate':                '#3498db',
    'Ecological':             '#2ecc71',
    'Healthcare/Surveillance':'#e74c3c',
    'Socioeconomic':          '#9b59b6',
    'Engineered':             '#f39c12'
}
feat_df['category'] = feat_df['feature'].apply(categorise)
colors_fi = [cat_colors[c] for c in feat_df['category']]

fig, ax = plt.subplots(figsize=(10, 12))
bars = ax.barh(feat_df['feature'], feat_df['importance'], color=colors_fi, edgecolor='white')
ax.set_xlabel('Feature Importance (XGBoost Gain)', fontsize=12)
ax.set_title('Figure 9: XGBoost Feature Importance\nby Ecological Category', fontsize=13, fontweight='bold')

patches = [mpatches.Patch(color=v, label=k) for k, v in cat_colors.items()]
ax.legend(handles=patches, loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig('fig9_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 5 most important features:")
print(feat_df.tail(5)[['feature','importance','category']].to_string(index=False))


In [ ]:
pivot = df.groupby(['country','host_species'])['outbreak_risk_level'].apply(
    lambda x: (x=='High').mean() * 100).unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            ax=ax, linewidths=0.5, cbar_kws={'label': '% High Risk Observations'})
ax.set_title('Figure 10: High-Risk Rate (%) by Country × Wildlife Host Species',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Wildlife Host Species')
ax.set_ylabel('Country')
plt.tight_layout()
plt.savefig('fig10_risk_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 8: Outbreak Risk Simulation

Simulating what happens to predicted risk when specific public health interventions are applied.

In [ ]:
# Take a high-risk scenario and simulate targeted public health interventions

# Baseline: high-risk profile
baseline = {
    'mean_temperature_c': 30.0, 'rainfall_mm': 220.0, 'humidity_pct': 85.0,
    'human_pop_density_km2': 300.0, 'deforestation_rate_pct': 9.5,
    'livestock_density': 90.0, 'wildlife_trade_index': 8.5,
    'healthcare_index': 22.0, 'surveillance_coverage': 0.18,
    'proximity_to_forest_km': 12.0, 'prev_outbreaks_5yr': 4,
    'travel_connectivity': 60.0, 'wet_season': 1,
    'sanitation_index': 18.0, 'month': 7,
    'country_enc': 5, 'host_enc': 0, 'disease_enc': 0,
    'temp_x_rain': 30.0*220/1000, 'deforest_x_pop': 9.5*300/100,
    'surveillance_deficit': (1-0.18)*(1-22/100),
    'wildlife_access_score': 8.5*(500-12)/500
}

scenarios = {
    'Baseline (No Intervention)': baseline.copy(),
    'Improved Surveillance (+40%)': {**baseline, 'surveillance_coverage': 0.58,
        'surveillance_deficit': (1-0.58)*(1-22/100)},
    'Healthcare Strengthened': {**baseline, 'healthcare_index': 55.0,
        'surveillance_deficit': (1-0.18)*(1-55/100)},
    'Deforestation Reduced (–60%)': {**baseline, 'deforestation_rate_pct': 3.8,
        'deforest_x_pop': 3.8*300/100},
    'Wildlife Trade Banned': {**baseline, 'wildlife_trade_index': 0.5,
        'wildlife_access_score': 0.5*(500-12)/500},
    'Combined Interventions': {**baseline,
        'surveillance_coverage': 0.70, 'healthcare_index': 60.0,
        'deforestation_rate_pct': 2.5, 'wildlife_trade_index': 1.0,
        'surveillance_deficit': (1-0.70)*(1-60/100),
        'deforest_x_pop': 2.5*300/100,
        'wildlife_access_score': 1.0*(500-12)/500}
}

risk_map = {0: 'High', 1: 'Low', 2: 'Medium'}  # Based on LabelEncoder output
risk_order_score = {'Low': 0, 'Medium': 1, 'High': 2}

print(f"{'Scenario':<40} {'Predicted Risk':<15} {'Probabilities (H/L/M)'}")
print("─" * 80)
sim_results = {}
for name, sc in scenarios.items():
    x_sc = np.array([[sc[f] for f in FEATURE_COLS]])
    pred_class = xgb_model.predict(x_sc)[0]
    pred_proba = xgb_model.predict_proba(x_sc)[0]
    risk_label = risk_map[pred_class]
    sim_results[name] = risk_label
    print(f"{name:<40} {risk_label:<15} H:{pred_proba[0]:.2f} L:{pred_proba[1]:.2f} M:{pred_proba[2]:.2f}")


In [ ]:
labels = list(scenarios.keys())
high_probs = []
for sc in scenarios.values():
    x_sc = np.array([[sc[f] for f in FEATURE_COLS]])
    high_probs.append(xgb_model.predict_proba(x_sc)[0][0])  # index 0 = High

fig, ax = plt.subplots(figsize=(12, 6))
bar_colors = ['#e74c3c'] + ['#3498db']*4 + ['#2ecc71']
bars = ax.barh(labels, high_probs, color=bar_colors, edgecolor='white', linewidth=1.5)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Risk Threshold 0.50')
for bar, val in zip(bars, high_probs):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2.,
            f'{val:.2f}', va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Predicted Probability of High Outbreak Risk')
ax.set_title('Figure 11: ZooPredict Intervention Simulation\nImpact of Public Health Interventions on Outbreak Probability',
             fontsize=12, fontweight='bold')
ax.set_xlim(0, 1.0)
ax.legend()
plt.tight_layout()
plt.savefig('fig11_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPolicy Insight:")
print("   Combined interventions (surveillance + healthcare + deforestation control + trade ban)")
print("   reduce High-risk probability most dramatically, demonstrating the importance")
print("   of multi-sector One Health approaches over single-intervention strategies.")


## Section 9: Conclusions

### Summary

| Model | Test Accuracy | Notes |
|---|---|---|
| Logistic Regression | Baseline | Good calibration, weak on High class |
| Random Forest | Strong | Robust, interpretable importances |
| Gradient Boosting | Strong | Stable across folds |
| **XGBoost** | **Best** | Highest AUC, best High-risk recall |

### Main takeaways

1. Healthcare and surveillance capacity are the most actionable predictors (unlike climate, they're policy-modifiable)
2. The deforestation x population density interaction ranked highly — habitat disruption near dense populations increases spillover risk
3. Bat and Primate hosts show disproportionately high-risk profiles, consistent with their role as reservoirs for Ebola, Marburg, and Nipah
4. Wet season months show elevated risk, suggesting seasonal surveillance intensification would help
5. No single intervention is enough — combined approaches reduce risk the most

### Limitations

- Synthetic dataset — needs validation on real WHO/PREDICT data
- No spatial autocorrelation modelling (neighbouring countries likely share risk)
- No temporal lagging — preceding months should inform current risk
- SHAP values would improve individual prediction explanations

### Next steps

- Integrate real WHO GOARN data when available
- Add LSTM for temporal trajectory prediction
- Sub-national (district-level) risk mapping
- Deploy as a REST API for real-time queries

---

> This project brought together ecology from my Zoology degree and the ML skills I've built since. Zoonotic outbreaks are a real problem for families across West and Central Africa — if tools like this can help a surveillance team catch a risk zone earlier, the work matters.
>
> — Karimat Abolarinwa

---

### References

1. Olival, K. J., et al. (2017). Host and viral traits predict zoonotic spillover from mammals. *Nature*, 546(7660), 646-650.
2. Jones, K. E., et al. (2008). Global trends in emerging infectious diseases. *Nature*, 451(7181), 990-993.
3. Keesing, F., et al. (2010). Impacts of biodiversity on the emergence and transmission of infectious diseases. *Nature*, 468(7324), 647-652.
4. Mordecai, E. A., et al. (2019). Thermal biology of mosquito-borne disease. *Ecology Letters*, 22(10), 1690-1708.
5. Murray, K. A., et al. (2015). Zoonoses of highest concern in Sub-Saharan Africa. *PLOS Neglected Tropical Diseases*, 9(8).
6. Karesh, W. B., et al. (2012). Ecology of zoonoses: natural and unnatural histories. *The Lancet*, 380(9857), 1936-1945.

In [ ]:
# Final model summary
print("=" * 60)
print("ZooPredict — Final Model Performance Summary")
print("=" * 60)
print(f"Best model       : XGBoost Classifier")
print(f"Test Accuracy    : {xgb_acc:.4f}")
print(f"ROC-AUC (OvR)    : {xgb_auc:.4f}")
print(f"Training records : {X_train.shape[0]:,}")
print(f"Test records     : {X_test.shape[0]:,}")
print(f"Feature count    : {X.shape[1]} (incl. 4 engineered interaction features)")
print(f"Target classes   : {list(le_target.classes_)}")
print()
print("XGBoost Classification Report (Test Set):")
print(classification_report(y_test, xgb_pred, target_names=le_target.classes_, digits=4))
print("=" * 60)
print("ZooPredict pipeline complete.")
